# Hierarchical NLI (E-first) — CafeBERT — Kaggle Training Notebook

Clone repo từ GitHub, cài deps, nạp secrets từ Kaggle Secrets, **in toàn bộ config trước khi chạy**, kéo data thật **ViANLI** (uitnlp/ViANLI trên HF Hub), rồi chạy 2 training run thật (Flat + Hierarchical CafeBERT) với W&B tracking và lưu trữ bắt buộc lên Hugging Face Hub (private repo).

**Mỗi cell train/eval đều check exit code và raise lỗi rõ ràng ngay nếu fail** — nếu 1 cell báo lỗi, đừng chạy tiếp cell sau, cuộn lên xem traceback ngay phía trên lỗi đó để biết nguyên nhân thật.

**Trước khi chạy, bắt buộc:**
1. `Add-ons` → `Secrets` → thêm 2 secret đúng tên:
   - `WANDB_API_KEY`
   - `HF_TOKEN`
   (giá trị là 2 key bạn đã cấp trước đó cho W&B và Hugging Face). Notebook sẽ fail rõ ràng ở bước nạp secrets nếu thiếu.
2. Settings (panel phải) → bật **Internet: On** và chọn **Accelerator: GPU T4 x2** (không dùng P100 — torch mới nhất đã bỏ hỗ trợ kiến trúc Pascal; code chỉ dùng 1 GPU nên dù chọn x2 cũng không lãng phí gì).
3. (Tùy chọn) đổi `DEBUG = True` ở cell bên dưới để chạy thử trên subset 200 mẫu trước khi chạy full — khuyến nghị làm trước để xác nhận toàn bộ pipeline chạy được trước khi đốt thời gian GPU cho full run.

In [ ]:
# --- GPU check ---
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(out or "⚠️ No GPU detected — vào Settings (panel phải) đổi Accelerator sang GPU rồi Restart session.")

In [ ]:
# --- Clone toàn bộ code từ GitHub ---
import os, pathlib

REPO_URL = "https://github.com/trantranuit/hierarchical-nli-e-first.git"
REPO_DIR = "/kaggle/working/hierarchical-nli-e-first"

if pathlib.Path(REPO_DIR).exists():
    !rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
assert _exit_code == 0, "git clone FAILED — kiểm tra Internet: On trong Settings."
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!ls -la

In [ ]:
# --- Cài dependencies (torch KHÔNG nằm trong requirements.txt — giữ nguyên
# bản torch Kaggle đã cài sẵn, đã khớp với GPU của instance này) ---
!pip install -q -r requirements.txt
assert _exit_code == 0, "pip install FAILED — xem traceback ở trên."

In [ ]:
# --- Sanity check torch/CUDA — phát hiện sớm nếu torch không khớp compute capability của GPU ---
import torch
print("torch:", torch.__version__, "| built for CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    cap = f"sm_{props.major}{props.minor}"
    supported = torch.cuda.get_arch_list()
    print(f"GPU: {props.name} (compute capability {cap})")
    print("torch supports archs:", supported)
    if cap not in supported:
        raise RuntimeError(
            f"{cap} KHÔNG nằm trong danh sách torch hỗ trợ {supported} → training sẽ crash với "
            f"'CUDA error: no kernel image is available'. Đổi Accelerator sang GPU T4 x2 (Settings > "
            f"Accelerator), Restart session, rồi chạy lại từ đầu."
        )
    print("✓ GPU/torch tương thích, sẵn sàng train.")
else:
    raise RuntimeError("CUDA không available — kiểm tra Settings > Accelerator (phải chọn GPU, không phải None).")

In [ ]:
# --- Nạp secrets từ Kaggle Secrets (KHÔNG bao giờ in giá trị thật ra output) ---
from kaggle_secrets import UserSecretsClient
import os

REQUIRED_SECRETS = ["WANDB_API_KEY", "HF_TOKEN"]

secrets_client = UserSecretsClient()
loaded, missing = {}, []
for key in REQUIRED_SECRETS:
    try:
        loaded[key] = secrets_client.get_secret(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Thiếu Kaggle secret(s): {missing}. "
        f"Vào Add-ons > Secrets, thêm đúng tên {missing}, rồi Save Version / chạy lại."
    )

# ghi ra .env để src/utils/env.py (và các training script chạy qua subprocess) tự động nạp
with open(".env", "w") as f:
    for k, v in loaded.items():
        f.write(f"{k}={v}\n")
        os.environ[k] = v

print("✓ Secrets loaded:", {k: f"***len={len(v)}" for k, v in loaded.items()})

In [ ]:
# --- IN TOÀN BỘ CONFIG TRƯỚC KHI CHẠY BẤT KỲ TRAINING NÀO ---
import yaml, json

with open("configs/config.yaml", encoding="utf-8") as f:
    cfg_text = f.read()
cfg = yaml.safe_load(cfg_text)

print("=" * 80)
print("FULL CONFIG — configs/config.yaml (raw)")
print("=" * 80)
print(cfg_text)
print("=" * 80)
print("Parsed config (sanity check):")
print(json.dumps(cfg, indent=2, ensure_ascii=False))
print("=" * 80)
print(f"W&B project : {cfg['wandb']['project']}  (entity theo API key)")
print(f"HF flat repo: {cfg['hf_hub']['flat_repo_id']}")
print(f"HF hier repo: {cfg['hf_hub']['hier_repo_id']}")

## Data — kéo thật **ViANLI** (thay cho synthetic placeholder trong repo)
`uitnlp/ViANLI` trên HF Hub (train 8012 / dev 1000 / test 1000, không gated). `scripts/prepare_vianli.py` tải `vianli_{train,dev,test}.jsonl`, đổi tên field (`uid→id`) và map nhãn (`entailment→E, contradiction→C, neutral→N`) rồi ghi đè `data/raw/*.jsonl`. Sau bước này, `load_or_generate()` trong `src/data/dataset.py` sẽ thấy file đã có và **không** sinh synthetic nữa — các script train dùng đúng data ViANLI này.

In [ ]:
!python3 scripts/prepare_vianli.py --config configs/config.yaml
assert _exit_code == 0, "prepare_vianli.py FAILED — xem traceback ở trên (thường do mất mạng hoặc HF Hub rate limit)."
print()
!wc -l data/raw/train.jsonl data/raw/dev.jsonl data/raw/test.jsonl
!head -n 2 data/raw/train.jsonl

In [ ]:
# --- (Tùy chọn) DEBUG = True để chạy thử trên 200 mẫu/split cho nhanh trước khi chạy full ---
DEBUG = False
debug_flag = "--debug" if DEBUG else ""
print("DEBUG mode:", DEBUG)

## Step 1 & 3 — 2 training runs thật (Flat + Hierarchical) trên ViANLI
Mỗi run tự động: train → log metrics/loss/confusion matrix lên **W&B** → predict dev/test → push checkpoint + tokenizer + predictions per-sample lên **HF Hub private repo** → log lại `hf_repo_id`/`hf_revision` vào W&B run summary.

**Quan trọng:** nếu 1 trong 2 cell dưới báo `AssertionError` — nghiĩa là training thật sự đã crash. Đừng chạy tiếp các cell sau (sẽ chỉ ra lỗi `FileNotFoundError` mớ hồ vì chưa có prediction nào được tạo) — cuộn lên xem traceback Python ngay phía trên dòng `assert` để biết lỗi thật (thường là OOM GPU, lỗi tải model, hoặc lỗi data).

In [ ]:
# --- Step 1: Flat CafeBERT (Run 1) ---
!python3 -m src.training.train_flat --config configs/config.yaml {debug_flag}
assert _exit_code == 0, "train_flat.py FAILED — cuộn lên xem traceback Python ở trên, KHÔNG chạy các cell sau cho đến khi fix."

In [ ]:
# --- Step 3: Hierarchical CafeBERT 2-head (Run 2) ---
!python3 -m src.training.train_hierarchical --config configs/config.yaml {debug_flag}
assert _exit_code == 0, "train_hierarchical.py FAILED — cuộn lên xem traceback Python ở trên, KHÔNG chạy các cell sau cho đến khi fix."

## Steps 2, 4A, 4B, 5 — offline (diagnostic, Hard/Soft, so sánh cuối)
Chạy offline từ prediction CSV đã lưu local (không train lại).

In [ ]:
!python3 scripts/evaluate_offline.py --config configs/config.yaml
assert _exit_code == 0, "evaluate_offline.py FAILED — cuộn lên xem traceback ở trên."

In [ ]:
# --- In bảng so sánh cuối cùng ---
report_path = pathlib.Path("outputs/comparison/comparison_report.md")
if not report_path.exists():
    raise FileNotFoundError(
        f"{report_path} chưa tồn tại — nghĩa là evaluate_offline.py không đủ cả 2 file predictions "
        f"(outputs/predictions/flat/*.csv và outputs/predictions/hier/*.csv) để so sánh. Quay lại 2 cell "
        f"train_flat / train_hierarchical ở trên, xem có in ra dòng '[predict dev] -> ...csv' / "
        f"'[predict test] -> ...csv' không — nếu không có, training đã crash trước khi tới bước predict."
    )
with open(report_path, encoding="utf-8") as f:
    print(f.read())

## Kết quả & nơi xem lại
- **W&B**: metrics/loss/confusion matrix của từng run — xem URL in ra trong log của mỗi lệnh train ở trên (dạng `https://wandb.ai/<entity>/hierarchical-nli-e-first/runs/<id>`).
- **Hugging Face Hub** (private): checkpoint + tokenizer/config + predictions per-sample (đã train trên ViANLI thật)
  - `https://huggingface.co/trinhtrantran122/hier-nli-e-first-flat-cafebert`
  - `https://huggingface.co/trinhtrantran122/hier-nli-e-first-hier-cafebert`
- `outputs/comparison/comparison_report.md` trong `/kaggle/working/hierarchical-nli-e-first/` — tải về từ tab Output của Kaggle nếu cần.